In [ ]:
import pandas as pd 
import numpy as np 
import kagglehub

In [ ]:
import pandas as pd
import os

DATA_PATH = "/kaggle/input/competitions/rsna-knee-abnormality-detection"

train = pd.read_csv(f"{DATA_PATH}/train.csv")
train_series = pd.read_csv(f"{DATA_PATH}/train_series.csv")

test = pd.read_csv(f"{DATA_PATH}/test.csv")
test_series = pd.read_csv(f"{DATA_PATH}/test_series.csv")

print("train:", train.shape)
print("train_series:", train_series.shape)

print("test:", test.shape)
print("test_series:", test_series.shape)

print("\nTrain columns:")
print(train.columns.tolist())

print("\nSeries columns:")
print(train_series.columns.tolist())

In [ ]:
train.tail()

In [ ]:
train_series.head()

In [ ]:
target_cols = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

train[target_cols].mean().sort_values()

In [ ]:
train.isna().sum()

In [ ]:
train.shape

In [ ]:
train[target_cols].notna().sum()

In [ ]:
train["Report"].head()

In [ ]:
labeled = train[train[target_cols].notna().all(axis=1)]

print("Labeled studies:", len(labeled))
print("Unlabeled studies:", len(train) - len(labeled))

In [ ]:
labeled[target_cols].mean().sort_values()

In [ ]:
labeled[target_cols].sum().sort_values()

In [ ]:
# unlabeled = train[train[target_cols].isna().all(axis=1)]

# for i in range(5):
#     print("=" * 100)
#     print(unlabeled.iloc[i]["Report"])

In [ ]:
from collections import Counter

# crude first check using common language indicators
reports = train["Report"].dropna()

for keyword in ["No hay", "Aucune", "Técnica", "MRI", "Knie", "Genou"]:
    print(keyword, reports.str.contains(keyword, case=False, na=False).sum())

In [ ]:
train["Report"].isna().sum()

In [ ]:
!pip install -q langdetect

In [ ]:
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 42

def detect_language(text):
    try:
        return detect(text)
    except:
        return "unknown"

train["language"] = train["Report"].apply(detect_language)

train["language"].value_counts()

In [ ]:
# labeled = train[train[target_cols].notna().all(axis=1)]

# for i in range(3):
#     print("=" * 100)
#     print(labeled.iloc[i]["Report"])
#     print()
#     print(labeled.iloc[i][target_cols])

In [ ]:
effusion_terms = [
    "effusion",
    "derrame",
    "épanchement",
    "joint fluid",
    "liquid"
]

for term in effusion_terms:
    mask = labeled["Report"].str.contains(
        term,
        case=False,
        na=False
    )
    
    print(term, mask.sum())

In [ ]:
terms = ["effusion", "derrame", "épanchement", "joint fluid", "liquid"]

mask = labeled["Report"].str.contains(
    "|".join(terms),
    case=False,
    na=False
)

print(
    labeled.loc[mask, ["Report", "Effusion"]]
    [["Effusion"]]
    .value_counts()
)

In [ ]:
# print(
#     labeled.loc[
#         mask & (labeled["Effusion"] == 0),
#         ["Report", "Effusion"]
#     ].to_string(index=False)
# )

In [ ]:
pd.crosstab(
    labeled["Effusion"],
    labeled["Report"].str.contains(
        "effusion|derrame|épanchement|joint fluid|liquid",
        case=False,
        na=False
    )
)

In [ ]:
import os

path = "/kaggle/input/datasets/pilkwang/rsna-knee-llm-labels"

for root, dirs, files in os.walk(path):
    print(root)
    for f in files:
        print("   ", f)

In [ ]:
labels = pd.read_csv(
    "/kaggle/input/datasets/pilkwang/rsna-knee-llm-labels/report_labels_v2.csv"
)

print(labels.shape)
print(labels.columns.tolist())
display(labels.head())

In [ ]:
print(labels.dtypes)

In [ ]:
merged = train.merge(
    labels,
    on="StudyInstanceUID",
    how="inner",
    suffixes=("_gold", "_llm")
)

print(merged.shape)

In [ ]:
print(
    merged[["StudyInstanceUID", "ACL_gold", "ACL__conf", "ACL__verdict"]]
    .head()
)

In [ ]:
gold = merged[merged["ACL_gold"].notna()]

print("Gold ACL studies:", len(gold))

print(
    pd.crosstab(
        gold["ACL_gold"],
        gold["ACL__verdict"]
    )
)

In [ ]:
for target in target_cols:
    gold = merged[merged[f"{target}_gold"].notna()]

    print(f"\n{'='*50}")
    print(target)
    print(pd.crosstab(
        gold[f"{target}_gold"],
        gold[f"{target}__verdict"]
    ))

In [ ]:
gold = merged[merged["ACL_gold"].notna()]

print(
    gold.groupby("ACL_gold")["ACL_llm"].describe()
)

In [ ]:
sample = pd.read_csv(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection/sample_submission.csv"
)

for col in target_cols:
    sample[col] = train[col].mean()

sample.to_csv("/kaggle/working/submission_baseline.csv", index=False)

sample.head()

In [ ]:
gold_studies = (
    train[
        train[target_cols].notna().any(axis=1)
    ]["StudyInstanceUID"]
    .dropna()
    .unique()
    .tolist()
)

print("Gold studies:", len(gold_studies))

In [ ]:

target_cols = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

# Your 58 gold studies
gold = train[
    train["StudyInstanceUID"].isin(gold_studies)
].copy()

print("Gold studies:", len(gold))

In [ ]:
from sklearn.metrics import f1_score

def evaluate_llm_labels(
    label_df,
    name,
    id_col="StudyInstanceUID"
):
    
    merged = gold[
        [id_col] + target_cols
    ].merge(
        label_df,
        on=id_col,
        how="inner",
        suffixes=("_gold", "_llm")
    )

    print(f"\n{'='*60}")
    print(name)
    print("Matched studies:", len(merged))

    scores = {}

    for target in target_cols:

        gold_col = f"{target}_gold"

        # Prefer explicit verdict column
        verdict_col = f"{target}__verdict"

        if verdict_col not in merged.columns:
            print(f"{target}: verdict column missing")
            continue

        pred = merged[verdict_col]

        # Only evaluate definite YES / NO
        mask = pred.isin(["YES", "NO"])

        y_true = merged.loc[mask, gold_col].astype(int)
        y_pred = (pred.loc[mask] == "YES").astype(int)

        if len(y_true) == 0:
            scores[target] = np.nan
            continue

        scores[target] = f1_score(
            y_true,
            y_pred,
            zero_division=0
        )

    result = pd.Series(scores)

    print("\nPer-target F1:")
    print(result.sort_values())

    print("\nMacro F1:",
          result.mean())

    return result, merged

In [ ]:
llm1 = pd.read_csv("/kaggle/input/datasets/lixin73/rsna-knee-llm-report-labels-sol56/report_labels_gpt56sol.csv")
llm2 = pd.read_csv("/kaggle/input/datasets/pilkwang/rsna-knee-llm-labels/report_labels_v2.csv")
llm3 = pd.read_csv("/kaggle/input/datasets/stevenleehans/rsna-knee-llm-report-labels/llm_labels_full.csv")

f1_1, m1 = evaluate_llm_labels(
    llm1,
    "LLM Dataset 1"
)

f1_2, m2 = evaluate_llm_labels(
    llm2,
    "LLM Dataset 2"
)

f1_3, m3 = evaluate_llm_labels(
    llm3,
    "LLM Dataset 3"
)

In [ ]:
comparison = pd.DataFrame({
    "LLM_1": f1_1,
    "LLM_2": f1_2,
    "LLM_3": f1_3
})

display(comparison)

print("\nMACRO F1")
print(comparison.mean().sort_values(ascending=False))

In [ ]:
print("LLM 1 columns:")
print(llm1.columns.tolist())

print("\n" + "="*80)

print("LLM 2 columns:")
print(llm2.columns.tolist())

print("\n" + "="*80)

print("LLM 3 columns:")
print(llm3.columns.tolist())

In [ ]:
for name, df in [
    ("LLM1", llm1),
    ("LLM2", llm2),
    ("LLM3", llm3)
]:
    print("\n" + "="*80)
    print(name)
    print("Shape:", df.shape)
    print(df.head(2).to_string())

In [ ]:
for name, df in [
    ("LLM1", llm1),
    ("LLM2", llm2),
    ("LLM3", llm3)
]:
    print("\n" + "="*80)
    print(name)

    for target in target_cols:
        verdict_col = f"{target}__verdict"

        if verdict_col in df.columns:
            print(
                target,
                "->",
                df[verdict_col].value_counts(dropna=False).to_dict()
            )

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import numpy as np
import pandas as pd


def evaluate_llm1(label_df, name="LLM1"):

    merged = gold[
        ["StudyInstanceUID"] + target_cols
    ].merge(
        label_df,
        on="StudyInstanceUID",
        how="inner",
        suffixes=("_gold", "_llm")
    )

    print("\n" + "="*70)
    print(name)
    print("Matched studies:", len(merged))

    results = []

    for target in target_cols:

        y_true = merged[f"{target}_gold"]
        y_pred = merged[f"{target}_llm"]

        # Only studies having an actual gold label
        mask = y_true.notna() & y_pred.notna()

        y_true = y_true[mask].astype(int)
        y_pred = y_pred[mask].astype(int)

        results.append({
            "Abnormality": target,
            "N": len(y_true),
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(
                y_true, y_pred, zero_division=0
            ),
            "Recall": recall_score(
                y_true, y_pred, zero_division=0
            ),
            "F1": f1_score(
                y_true, y_pred, zero_division=0
            )
        })

    result = pd.DataFrame(results)

    print("\nResults:")
    display(result)

    print("Macro F1:", result["F1"].mean())

    return result, merged

In [ ]:
def evaluate_llm2(label_df, name="LLM2"):

    merged = gold[
        ["StudyInstanceUID"] + target_cols
    ].merge(
        label_df,
        on="StudyInstanceUID",
        how="inner",
        suffixes=("_gold", "_llm")
    )

    print("\n" + "="*70)
    print(name)
    print("Matched studies:", len(merged))

    results = []

    for target in target_cols:

        gold_col = f"{target}_gold"
        verdict_col = f"{target}__verdict"

        y_true_all = merged[gold_col]
        pred_all = merged[verdict_col]

        # Gold labels that exist
        gold_mask = y_true_all.notna()

        total_gold = gold_mask.sum()

        # Only definite YES / NO
        mask = (
            gold_mask
            & pred_all.isin(["YES", "NO"])
        )

        y_true = y_true_all[mask].astype(int)
        y_pred = (
            pred_all[mask] == "YES"
        ).astype(int)

        results.append({
            "Abnormality": target,
            "Gold_N": total_gold,
            "Evaluated_N": len(y_true),
            "Coverage": len(y_true) / total_gold
                if total_gold > 0 else np.nan,
            "Accuracy": accuracy_score(y_true, y_pred)
                if len(y_true) > 0 else np.nan,
            "Precision": precision_score(
                y_true, y_pred, zero_division=0
            ) if len(y_true) > 0 else np.nan,
            "Recall": recall_score(
                y_true, y_pred, zero_division=0
            ) if len(y_true) > 0 else np.nan,
            "F1": f1_score(
                y_true, y_pred, zero_division=0
            ) if len(y_true) > 0 else np.nan
        })

    result = pd.DataFrame(results)

    display(result)

    print("Macro F1:", result["F1"].mean())

    return result, merged

In [ ]:
for target in target_cols:

    print("\n" + "="*50)
    print(target)

    print("LLM1:")
    print(llm1[target].value_counts(dropna=False).head(20))

    print("\nLLM3:")
    print(llm3[target].value_counts(dropna=False).head(20))

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import numpy as np
import pandas as pd


def evaluate_llm1(label_df, name="LLM1"):

    merged = gold[
        ["StudyInstanceUID"] + target_cols
    ].merge(
        label_df,
        on="StudyInstanceUID",
        how="inner",
        suffixes=("_gold", "_llm")
    )

    print("\n" + "="*70)
    print(name)
    print("Matched studies:", len(merged))

    results = []

    for target in target_cols:

        gold_col = f"{target}_gold"
        pred_col = f"{target}_llm"

        gold_values = merged[gold_col]
        pred_values = merged[pred_col]

        gold_mask = gold_values.notna()

        total_gold = gold_mask.sum()

        # Only definite 0 or 1 predictions
        mask = (
            gold_mask
            & pred_values.isin([0, 1])
        )

        y_true = gold_values[mask].astype(int)
        y_pred = pred_values[mask].astype(int)

        results.append({
            "Abnormality": target,
            "Gold_N": total_gold,
            "Evaluated_N": len(y_true),
            "Coverage": len(y_true) / total_gold
                if total_gold > 0 else np.nan,
            "Accuracy": accuracy_score(y_true, y_pred)
                if len(y_true) > 0 else np.nan,
            "Precision": precision_score(
                y_true, y_pred, zero_division=0
            ) if len(y_true) > 0 else np.nan,
            "Recall": recall_score(
                y_true, y_pred, zero_division=0
            ) if len(y_true) > 0 else np.nan,
            "F1": f1_score(
                y_true, y_pred, zero_division=0
            ) if len(y_true) > 0 else np.nan
        })

    result = pd.DataFrame(results)

    display(result)

    print("Macro F1:", result["F1"].mean())
    print("Mean Coverage:", result["Coverage"].mean())

    return result, merged

In [ ]:
def evaluate_llm2(label_df, name="LLM2"):

    merged = gold[
        ["StudyInstanceUID"] + target_cols
    ].merge(
        label_df,
        on="StudyInstanceUID",
        how="inner",
        suffixes=("_gold", "_llm")
    )

    print("\n" + "="*70)
    print(name)
    print("Matched studies:", len(merged))

    results = []

    for target in target_cols:

        gold_col = f"{target}_gold"
        verdict_col = f"{target}__verdict"

        gold_values = merged[gold_col]
        pred_values = merged[verdict_col]

        gold_mask = gold_values.notna()

        total_gold = gold_mask.sum()

        # Only definite YES / NO
        mask = (
            gold_mask
            & pred_values.isin(["YES", "NO"])
        )

        y_true = gold_values[mask].astype(int)

        y_pred = (
            pred_values[mask] == "YES"
        ).astype(int)

        results.append({
            "Abnormality": target,
            "Gold_N": total_gold,
            "Evaluated_N": len(y_true),
            "Coverage": len(y_true) / total_gold
                if total_gold > 0 else np.nan,
            "Accuracy": accuracy_score(y_true, y_pred)
                if len(y_true) > 0 else np.nan,
            "Precision": precision_score(
                y_true, y_pred, zero_division=0
            ) if len(y_true) > 0 else np.nan,
            "Recall": recall_score(
                y_true, y_pred, zero_division=0
            ) if len(y_true) > 0 else np.nan,
            "F1": f1_score(
                y_true, y_pred, zero_division=0
            ) if len(y_true) > 0 else np.nan
        })

    result = pd.DataFrame(results)

    display(result)

    print("Macro F1:", result["F1"].mean())
    print("Mean Coverage:", result["Coverage"].mean())

    return result, merged

In [ ]:
def evaluate_llm3(label_df, name="LLM3"):

    merged = gold[
        ["StudyInstanceUID"] + target_cols
    ].merge(
        label_df,
        on="StudyInstanceUID",
        how="inner",
        suffixes=("_gold", "_llm")
    )

    print("\n" + "="*70)
    print(name)
    print("Matched studies:", len(merged))

    results = []

    for target in target_cols:

        gold_col = f"{target}_gold"
        pred_col = f"{target}_llm"

        gold_values = merged[gold_col]
        pred_values = merged[pred_col]

        gold_mask = gold_values.notna()

        total_gold = gold_mask.sum()

        # LLM3:
        # < 0.5 = NO
        # > 0.5 = YES
        # exactly 0.5 = UNCERTAIN
        mask = (
            gold_mask
            & pred_values.notna()
            & (pred_values != 0.5)
        )

        y_true = gold_values[mask].astype(int)

        y_pred = (
            pred_values[mask] > 0.5
        ).astype(int)

        results.append({
            "Abnormality": target,
            "Gold_N": total_gold,
            "Evaluated_N": len(y_true),
            "Coverage": len(y_true) / total_gold
                if total_gold > 0 else np.nan,
            "Accuracy": accuracy_score(y_true, y_pred)
                if len(y_true) > 0 else np.nan,
            "Precision": precision_score(
                y_true, y_pred, zero_division=0
            ) if len(y_true) > 0 else np.nan,
            "Recall": recall_score(
                y_true, y_pred, zero_division=0
            ) if len(y_true) > 0 else np.nan,
            "F1": f1_score(
                y_true, y_pred, zero_division=0
            ) if len(y_true) > 0 else np.nan
        })

    result = pd.DataFrame(results)

    display(result)

    print("Macro F1:", result["F1"].mean())
    print("Mean Coverage:", result["Coverage"].mean())

    return result, merged

In [ ]:
result1, m1 = evaluate_llm1(
    llm1,
    "LLM Dataset 1"
)

result2, m2 = evaluate_llm2(
    llm2,
    "LLM Dataset 2"
)

result3, m3 = evaluate_llm3(
    llm3,
    "LLM Dataset 3"
)

In [ ]:
comparison = pd.DataFrame({
    "LLM_1_F1": result1.set_index("Abnormality")["F1"],
    "LLM_2_F1": result2.set_index("Abnormality")["F1"],
    "LLM_3_F1": result3.set_index("Abnormality")["F1"]
})

display(comparison)

In [ ]:
print("MACRO F1")
print(comparison.mean().sort_values(ascending=False))

In [ ]:
coverage_comparison = pd.DataFrame({
    "LLM_1_Coverage": result1.set_index("Abnormality")["Coverage"],
    "LLM_2_Coverage": result2.set_index("Abnormality")["Coverage"],
    "LLM_3_Coverage": result3.set_index("Abnormality")["Coverage"]
})

display(coverage_comparison)

print("\nAverage Coverage:")
print(
    coverage_comparison.mean()
    .sort_values(ascending=False)
)

In [ ]:
# Check how many predictions are actually correct
# across ALL gold-labelled studies.

def overall_agreement_llm1(label_df, name="LLM1"):

    merged = gold[
        ["StudyInstanceUID"] + target_cols
    ].merge(
        label_df,
        on="StudyInstanceUID",
        how="inner",
        suffixes=("_gold", "_llm")
    )

    results = []

    for target in target_cols:

        gold_col = f"{target}_gold"
        pred_col = f"{target}_llm"

        mask = (
            merged[gold_col].notna()
            & merged[pred_col].notna()
        )

        y_true = merged.loc[mask, gold_col].astype(int)
        pred = merged.loc[mask, pred_col]

        # Only exact 0/1 predictions count as usable.
        usable = pred.isin([0, 1])

        y_true = y_true[usable]
        y_pred = pred[usable].astype(int)

        correct = (y_true == y_pred).sum()
        total_gold = mask.sum()

        results.append({
            "Abnormality": target,
            "Gold_N": total_gold,
            "Correct": correct,
            "Usable": len(y_true),
            "Coverage": len(y_true) / total_gold,
            "Overall_Correct_Rate": correct / total_gold
        })

    result = pd.Data

In [ ]:
gold_ids = set(gold["StudyInstanceUID"])

llm1_ids = set(llm1["StudyInstanceUID"])
llm2_ids = set(llm2["StudyInstanceUID"])
llm3_ids = set(llm3["StudyInstanceUID"])

print("Missing from LLM1:")
print(gold_ids - llm1_ids)

print("\nMissing from LLM2:")
print(gold_ids - llm2_ids)

print("\nMissing from LLM3:")
print(gold_ids - llm3_ids)

In [ ]:
# Keep exactly the 58 gold studies
gold_eval = gold[
    ["StudyInstanceUID"] + target_cols
].copy()

# ---------------------------------------------------
# LLM 1
# ---------------------------------------------------

llm1_eval = llm1[
    ["StudyInstanceUID"] + target_cols
].copy()

llm1_eval = llm1_eval.rename(
    columns={
        col: f"{col}_LLM1"
        for col in target_cols
    }
)

# ---------------------------------------------------
# LLM 2
# ---------------------------------------------------

llm2_eval = llm2[
    ["StudyInstanceUID"] +
    [
        f"{target}__verdict"
        for target in target_cols
    ]
].copy()

llm2_eval = llm2_eval.rename(
    columns={
        f"{target}__verdict": f"{target}_LLM2"
        for target in target_cols
    }
)

# ---------------------------------------------------
# LLM 3
# ---------------------------------------------------

llm3_eval = llm3[
    ["StudyInstanceUID"] + target_cols
].copy()

llm3_eval = llm3_eval.rename(
    columns={
        col: f"{col}_LLM3"
        for col in target_cols
    }
)

# ---------------------------------------------------
# Merge everything
# ---------------------------------------------------

comparison_data = gold_eval.merge(
    llm1_eval,
    on="StudyInstanceUID",
    how="left"
)

comparison_data = comparison_data.merge(
    llm2_eval,
    on="StudyInstanceUID",
    how="left"
)

comparison_data = comparison_data.merge(
    llm3_eval,
    on="StudyInstanceUID",
    how="left"
)

print("Rows:", len(comparison_data))
print("Columns:", len(comparison_data.columns))

display(comparison_data.head())

In [ ]:
for target in target_cols:

    # -------------------------
    # Gold
    # -------------------------
    
    comparison_data[f"{target}_GOLD"] = (
        comparison_data[target]
        .map({
            0: "NO",
            1: "YES"
        })
    )

    # -------------------------
    # LLM 1
    # -------------------------
    
    comparison_data[f"{target}_LLM1_FINAL"] = (
        comparison_data[f"{target}_LLM1"]
        .map({
            0.0: "NO",
            1.0: "YES"
        })
    )

    # -------------------------
    # LLM 2
    # -------------------------
    
    comparison_data[f"{target}_LLM2_FINAL"] = (
        comparison_data[f"{target}_LLM2"]
    )

    # -------------------------
    # LLM 3
    # -------------------------
    
    comparison_data[f"{target}_LLM3_FINAL"] = np.nan

    comparison_data.loc[
        comparison_data[f"{target}_LLM3"] < 0.5,
        f"{target}_LLM3_FINAL"
    ] = "NO"

    comparison_data.loc[
        comparison_data[f"{target}_LLM3"] > 0.5,
        f"{target}_LLM3_FINAL"
    ] = "YES"

    comparison_data.loc[
        comparison_data[f"{target}_LLM3"] == 0.5,
        f"{target}_LLM3_FINAL"
    ] = "UNCERTAIN"

In [ ]:
agreement_results = []

for target in target_cols:

    gold_col = f"{target}_GOLD"

    for llm in ["LLM1", "LLM2", "LLM3"]:

        pred_col = f"{target}_{llm}_FINAL"

        temp = comparison_data[
            [gold_col, pred_col]
        ].copy()

        # Gold must exist
        temp = temp[temp[gold_col].notna()]

        # Only definite predictions
        temp = temp[
            temp[pred_col].isin(["YES", "NO"])
        ]

        correct = (
            temp[gold_col] == temp[pred_col]
        ).sum()

        evaluated = len(temp)

        agreement_results.append({
            "Abnormality": target,
            "LLM": llm,
            "Gold_N": comparison_data[gold_col].notna().sum(),
            "Evaluated_N": evaluated,
            "Correct": correct,
            "Incorrect": evaluated - correct,
            "Coverage": evaluated / comparison_data[gold_col].notna().sum(),
            "Agreement": correct / evaluated
                if evaluated > 0 else np.nan
        })

agreement_df = pd.DataFrame(agreement_results)

display(agreement_df)

In [ ]:
final_summary = []

for llm, result in [
    ("LLM1", result1),
    ("LLM2", result2),
    ("LLM3", result3)
]:

    final_summary.append({
        "LLM": llm,
        "Macro F1": result["F1"].mean(),
        "Mean Coverage": result["Coverage"].mean(),
        "Mean Accuracy": result["Accuracy"].mean(),
        "Mean Precision": result["Precision"].mean(),
        "Mean Recall": result["Recall"].mean()
    })

final_summary_df = pd.DataFrame(final_summary)

display(
    final_summary_df.sort_values(
        "Macro F1",
        ascending=False
    )
)

In [ ]:
best_llm = comparison.idxmax(axis=1)

best_llm = best_llm.replace({
    "LLM_1_F1": "LLM1",
    "LLM_2_F1": "LLM2",
    "LLM_3_F1": "LLM3"
})

per_abnormality_result = comparison.copy()

per_abnormality_result["Best LLM"] = best_llm
per_abnormality_result["Best F1"] = comparison.max(axis=1)

display(per_abnormality_result)

In [ ]:
# Check confidence columns from LLM1 and LLM2

print("LLM1 confidence examples:")
display(
    llm1[
        ["StudyInstanceUID", "ACL", "ACL__conf",
         "MCL", "MCL__conf"]
    ].head(10)
)

print("\nLLM2 confidence examples:")
display(
    llm2[
        ["StudyInstanceUID", "ACL", "ACL__conf", "ACL__verdict",
         "MCL", "MCL__conf", "MCL__verdict"]
    ].head(10)
)

print("\nLLM3 examples:")
display(
    llm3[
        ["StudyInstanceUID", "ACL", "MCL",
         "Synovitis", "Baker's", "Fracture"]
    ].head(10)
)

In [ ]:
import numpy as np
import pandas as pd

# Make copies
l1 = llm1.copy()
l2 = llm2.copy()
l3 = llm3.copy()

# ---------------------------------------------------------
# LLM1
# ---------------------------------------------------------
# Convert:
# label 0 + confidence 0.98 -> probability 0.02
# label 1 + confidence 0.98 -> probability 0.98

for target in target_cols:

    label = l1[target]
    conf = l1[f"{target}__conf"]

    l1[f"{target}__prob"] = np.where(
        label == 1,
        conf,
        np.where(label == 0, 1 - conf, np.nan)
    )

    l1[f"{target}__weight"] = conf


# ---------------------------------------------------------
# LLM2
# ---------------------------------------------------------
# YES -> probability = confidence
# NO  -> probability = 1 - confidence
# UNK -> uncertain, so don't use it
# ---------------------------------------------------------

for target in target_cols:

    verdict = l2[f"{target}__verdict"]
    conf = l2[f"{target}__conf"]

    l2[f"{target}__prob"] = np.where(
        verdict == "YES",
        conf,
        np.where(
            verdict == "NO",
            1 - conf,
            np.nan
        )
    )

    l2[f"{target}__weight"] = np.where(
        verdict.isin(["YES", "NO"]),
        conf,
        np.nan
    )


# ---------------------------------------------------------
# LLM3
# ---------------------------------------------------------
# LLM3 value is already a probability.
# 0.50 = uncertain.
# ---------------------------------------------------------

for target in target_cols:

    prob = l3[target]

    l3[f"{target}__prob"] = prob

    l3[f"{target}__weight"] = np.where(
        prob != 0.50,
        abs(prob - 0.50) * 2,
        np.nan
    )

In [ ]:
ensemble = gold[["StudyInstanceUID"]].copy()

ensemble = ensemble.merge(
    l1,
    on="StudyInstanceUID",
    how="left",
    suffixes=("", "_l1")
)

ensemble = ensemble.merge(
    l2,
    on="StudyInstanceUID",
    how="left",
    suffixes=("", "_l2")
)

ensemble = ensemble.merge(
    l3,
    on="StudyInstanceUID",
    how="left",
    suffixes=("", "_l3")
)

print("Studies in ensemble:", len(ensemble))

In [ ]:
for target in target_cols:

    prob1 = ensemble[f"{target}__prob"]
    weight1 = ensemble[f"{target}__weight"]

    prob2 = ensemble[f"{target}__prob_l2"]
    weight2 = ensemble[f"{target}__weight_l2"]

    prob3 = ensemble[f"{target}__prob_l3"]
    weight3 = ensemble[f"{target}__weight_l3"]

    # Weighted numerator
    numerator = (
        prob1.fillna(0) * weight1.fillna(0)
        + prob2.fillna(0) * weight2.fillna(0)
        + prob3.fillna(0) * weight3.fillna(0)
    )

    # Total available confidence
    denominator = (
        weight1.fillna(0)
        + weight2.fillna(0)
        + weight3.fillna(0)
    )

    ensemble[f"{target}__ensemble_prob"] = np.where(
        denominator > 0,
        numerator / denominator,
        np.nan
    )

In [ ]:
for target in target_cols:

    prob = ensemble[f"{target}__ensemble_prob"]

    ensemble[f"{target}__prediction"] = np.where(
        prob > 0.50,
        1,
        np.where(
            prob < 0.50,
            0,
            np.nan
        )
    )

In [ ]:
display(
    ensemble[
        [
            "StudyInstanceUID",
            "ACL__ensemble_prob",
            "ACL__prediction",
            "MCL__ensemble_prob",
            "MCL__prediction",
            "Synovitis__ensemble_prob",
            "Synovitis__prediction"
        ]
    ].head(20)
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

ensemble_results = []

for target in target_cols:

    # Gold labels
    gold_values = gold[target]

    # Ensemble predictions
    pred_values = ensemble[f"{target}__prediction"]

    # Only studies where gold label exists
    gold_mask = gold_values.notna()

    # Only definite ensemble predictions
    pred_mask = pred_values.notna()

    mask = gold_mask & pred_mask

    y_true = gold_values[mask].astype(int)
    y_pred = pred_values[mask].astype(int)

    total_gold = gold_mask.sum()
    evaluated = len(y_true)

    ensemble_results.append({
        "Abnormality": target,
        "Gold_N": total_gold,
        "Evaluated_N": evaluated,
        "Coverage": (
            evaluated / total_gold
            if total_gold > 0 else np.nan
        ),
        "Accuracy": (
            accuracy_score(y_true, y_pred)
            if evaluated > 0 else np.nan
        ),
        "Precision": (
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            )
            if evaluated > 0 else np.nan
        ),
        "Recall": (
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            )
            if evaluated > 0 else np.nan
        ),
        "F1": (
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            )
            if evaluated > 0 else np.nan
        )
    })

ensemble_results = pd.DataFrame(ensemble_results)

display(ensemble_results)

print("Macro F1:",
      ensemble_results["F1"].mean())

print("Mean Coverage:",
      ensemble_results["Coverage"].mean())

print("Mean Accuracy:",
      ensemble_results["Accuracy"].mean())

print("Mean Precision:",
      ensemble_results["Precision"].mean())

print("Mean Recall:",
      ensemble_results["Recall"].mean())

In [ ]:
# Create clean copies with ONLY the required columns

l1_clean = llm1[["StudyInstanceUID"]].copy()
l2_clean = llm2[["StudyInstanceUID"]].copy()
l3_clean = llm3[["StudyInstanceUID"]].copy()

for target in target_cols:

    # -----------------------------
    # LLM1
    # -----------------------------
    l1_clean[f"{target}_prob_1"] = np.where(
        llm1[target] == 1,
        llm1[f"{target}__conf"],
        np.where(
            llm1[target] == 0,
            1 - llm1[f"{target}__conf"],
            np.nan
        )
    )

    l1_clean[f"{target}_weight_1"] = np.where(
        llm1[target].isin([0, 1]),
        llm1[f"{target}__conf"],
        np.nan
    )


    # -----------------------------
    # LLM2
    # -----------------------------
    l2_clean[f"{target}_prob_2"] = np.where(
        llm2[f"{target}__verdict"] == "YES",
        llm2[f"{target}__conf"],
        np.where(
            llm2[f"{target}__verdict"] == "NO",
            1 - llm2[f"{target}__conf"],
            np.nan
        )
    )

    l2_clean[f"{target}_weight_2"] = np.where(
        llm2[f"{target}__verdict"].isin(["YES", "NO"]),
        llm2[f"{target}__conf"],
        np.nan
    )


    # -----------------------------
    # LLM3
    # -----------------------------
    l3_clean[f"{target}_prob_3"] = llm3[target]

    l3_clean[f"{target}_weight_3"] = np.where(
        llm3[target] != 0.50,
        abs(llm3[target] - 0.50) * 2,
        np.nan
    )


print("LLM1 clean shape:", l1_clean.shape)
print("LLM2 clean shape:", l2_clean.shape)
print("LLM3 clean shape:", l3_clean.shape)

In [ ]:
ensemble = gold[["StudyInstanceUID"]].copy()

ensemble = ensemble.merge(
    l1_clean,
    on="StudyInstanceUID",
    how="left"
)

ensemble = ensemble.merge(
    l2_clean,
    on="StudyInstanceUID",
    how="left"
)

ensemble = ensemble.merge(
    l3_clean,
    on="StudyInstanceUID",
    how="left"
)

print("Number of ensemble studies:", len(ensemble))

In [ ]:
for target in target_cols:

    p1 = ensemble[f"{target}_prob_1"]
    w1 = ensemble[f"{target}_weight_1"]

    p2 = ensemble[f"{target}_prob_2"]
    w2 = ensemble[f"{target}_weight_2"]

    p3 = ensemble[f"{target}_prob_3"]
    w3 = ensemble[f"{target}_weight_3"]

    # Replace missing weights with 0
    w1 = w1.fillna(0)
    w2 = w2.fillna(0)
    w3 = w3.fillna(0)

    # Replace missing probabilities with 0
    p1 = p1.fillna(0)
    p2 = p2.fillna(0)
    p3 = p3.fillna(0)

    numerator = (
        p1 * w1 +
        p2 * w2 +
        p3 * w3
    )

    denominator = (
        w1 +
        w2 +
        w3
    )

    ensemble[f"{target}_ensemble_prob"] = np.where(
        denominator > 0,
        numerator / denominator,
        np.nan
    )

In [ ]:
display(
    ensemble[
        [
            "StudyInstanceUID",

            "ACL_prob_1",
            "ACL_prob_2",
            "ACL_prob_3",
            "ACL_ensemble_prob",

            "MCL_prob_1",
            "MCL_prob_2",
            "MCL_prob_3",
            "MCL_ensemble_prob"
        ]
    ].head(10)
)

In [ ]:
for target in target_cols:

    prob = ensemble[f"{target}_ensemble_prob"]

    ensemble[f"{target}_prediction"] = np.where(
        prob > 0.50,
        1,
        np.where(
            prob < 0.50,
            0,
            np.nan
        )
    )

In [ ]:
for target in target_cols:

    predictions = ensemble[f"{target}_prediction"]

    print(
        target,
        "→",
        predictions.notna().sum(),
        "/ 58 definite predictions"
    )

In [ ]:
ensemble_results = []

for target in target_cols:

    # Get gold labels and ensemble predictions
    gold_values = gold[target].reset_index(drop=True)
    predictions = ensemble[f"{target}_prediction"].reset_index(drop=True)

    # Only evaluate where gold exists
    gold_mask = gold_values.notna()

    # Only evaluate definite ensemble predictions
    prediction_mask = predictions.notna()

    # Combine masks
    mask = gold_mask & prediction_mask

    # Use positional indexing
    y_true = gold_values[mask].astype(int)
    y_pred = predictions[mask].astype(int)

    total_gold = gold_mask.sum()
    evaluated = len(y_true)

    ensemble_results.append({
        "Abnormality": target,
        "Gold_N": total_gold,
        "Evaluated_N": evaluated,
        "Coverage": (
            evaluated / total_gold
            if total_gold > 0 else np.nan
        ),
        "Accuracy": (
            accuracy_score(y_true, y_pred)
            if evaluated > 0 else np.nan
        ),
        "Precision": (
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            )
            if evaluated > 0 else np.nan
        ),
        "Recall": (
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            )
            if evaluated > 0 else np.nan
        ),
        "F1": (
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            )
            if evaluated > 0 else np.nan
        )
    })


ensemble_results = pd.DataFrame(ensemble_results)

display(ensemble_results)

print("\nMacro F1:",
      ensemble_results["F1"].mean())

print("Mean Coverage:",
      ensemble_results["Coverage"].mean())

print("Mean Accuracy:",
      ensemble_results["Accuracy"].mean())

print("Mean Precision:",
      ensemble_results["Precision"].mean())

print("Mean Recall:",
      ensemble_results["Recall"].mean())

In [ ]:
# FINAL ABNORMALITY-WISE PREDICTOR

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Best LLM for each abnormality based on our evaluation
best_llm = {
    "ACL": "LLM1",
    "MCL": "LLM1",
    "Medial Meniscus": "LLM1",
    "Lateral Meniscus": "LLM1",
    "Medial OA": "LLM3",
    "Lateral OA": "LLM2",
    "PF OA": "LLM3",
    "Effusion": "LLM1",
    "Synovitis": "LLM2",
    "Baker's": "LLM3",
    "Contusion": "LLM1",
    "Fracture": "LLM2"
}

# Start with the 58 gold studies
final_pred = gold[["StudyInstanceUID"]].drop_duplicates().reset_index(drop=True)

for target in target_cols:

    # -------------------------
    # LLM1 prediction
    # -------------------------
    temp1 = llm1.set_index("StudyInstanceUID")[target]
    pred1 = temp1.reindex(final_pred["StudyInstanceUID"]).reset_index(drop=True)

    # Only exact 0/1 are definite predictions
    pred1 = pred1.where(pred1.isin([0, 1]), np.nan)

    # -------------------------
    # LLM2 prediction
    # -------------------------
    temp2 = llm2.set_index("StudyInstanceUID")[f"{target}__verdict"]
    pred2 = temp2.reindex(final_pred["StudyInstanceUID"]).reset_index(drop=True)

    pred2 = np.where(
        pred2 == "YES", 1,
        np.where(pred2 == "NO", 0, np.nan)
    )

    # -------------------------
    # LLM3 prediction
    # -------------------------
    temp3 = llm3.set_index("StudyInstanceUID")[target]
    pred3 = temp3.reindex(final_pred["StudyInstanceUID"]).reset_index(drop=True)

    pred3 = np.where(
        pred3 > 0.5, 1,
        np.where(pred3 < 0.5, 0, np.nan)
    )

    # -------------------------
    # Select best LLM
    # -------------------------
    if best_llm[target] == "LLM1":
        final_pred[target] = pred1

    elif best_llm[target] == "LLM2":
        final_pred[target] = pred2

    elif best_llm[target] == "LLM3":
        final_pred[target] = pred3


print("Final prediction dataframe:")
print(final_pred.head())

print("\nSelected LLM for each abnormality:")
for target in target_cols:
    print(f"{target:20s} -> {best_llm[target]}")

In [ ]:
# EVALUATE FINAL ABNORMALITY-WISE PREDICTOR

results = []

gold_eval = (
    gold[["StudyInstanceUID"] + target_cols]
    .drop_duplicates("StudyInstanceUID")
    .set_index("StudyInstanceUID")
    .reindex(final_pred["StudyInstanceUID"])
    .reset_index()
)

for target in target_cols:

    y_true = gold_eval[target]
    y_pred = final_pred[target]

    # Only evaluate definite predictions
    mask = y_true.notna() & pd.notna(y_pred)

    y_true_valid = y_true[mask].astype(int)
    y_pred_valid = pd.Series(y_pred[mask]).astype(int)

    results.append({
        "Abnormality": target,
        "Selected LLM": best_llm[target],
        "Gold N": len(y_true_valid),
        "Correct": (y_true_valid.values == y_pred_valid.values).sum(),
        "Incorrect": (y_true_valid.values != y_pred_valid.values).sum(),
        "Accuracy": accuracy_score(y_true_valid, y_pred_valid),
        "Precision": precision_score(y_true_valid, y_pred_valid, zero_division=0),
        "Recall": recall_score(y_true_valid, y_pred_valid, zero_division=0),
        "F1": f1_score(y_true_valid, y_pred_valid, zero_division=0)
    })

final_results = pd.DataFrame(results)

print(final_results.to_string(index=False))

print("\nMean Accuracy:", final_results["Accuracy"].mean())
print("Macro F1:", final_results["F1"].mean())

In [ ]:
# SHOW ALL INCORRECT PREDICTIONS

for target in target_cols:

    y_true = gold_eval[target]
    y_pred = final_pred[target]

    mask = y_true.notna() & pd.notna(y_pred)

    wrong = (
        (y_true[mask].astype(int) != y_pred[mask].astype(int))
    )

    wrong_studies = final_pred.loc[mask, "StudyInstanceUID"][wrong.values]

    print("\n" + "=" * 60)
    print(target)
    print("=" * 60)

    for study_id in wrong_studies:
        gold_value = int(y_true[mask][wrong].loc[
            y_true[mask][wrong].index[
                list(wrong_studies).index(study_id)
            ]
        ])
        pred_value = int(final_pred.loc[
            final_pred["StudyInstanceUID"] == study_id, target
        ].iloc[0])

        print(
            f"{study_id} | Gold: {gold_value} | Prediction: {pred_value}"
        )

In [ ]:
# CHECK WHETHER LLM2 OR LLM3 CAN FIX THE ERRORS
# Focus only on cases where our selected LLM was wrong.

for target in target_cols:

    selected = best_llm[target]

    # Get predictions from all 3 LLMs
    p1 = llm1.set_index("StudyInstanceUID")[target]
    p1 = p1.reindex(final_pred["StudyInstanceUID"]).reset_index(drop=True)
    p1 = p1.where(p1.isin([0, 1]), np.nan)

    p2 = llm2.set_index("StudyInstanceUID")[f"{target}__verdict"]
    p2 = p2.reindex(final_pred["StudyInstanceUID"]).reset_index(drop=True)
    p2 = np.where(
        p2 == "YES", 1,
        np.where(p2 == "NO", 0, np.nan)
    )

    p3 = llm3.set_index("StudyInstanceUID")[target]
    p3 = p3.reindex(final_pred["StudyInstanceUID"]).reset_index(drop=True)
    p3 = np.where(
        p3 > 0.5, 1,
        np.where(p3 < 0.5, 0, np.nan)
    )

    gold_values = gold_eval[target].reset_index(drop=True)

    selected_pred = final_pred[target].reset_index(drop=True)

    print("\n" + "=" * 80)
    print(target, " | Selected:", selected)
    print("=" * 80)

    for i in range(len(final_pred)):

        if pd.isna(gold_values[i]) or pd.isna(selected_pred[i]):
            continue

        # Selected LLM made a wrong prediction
        if int(selected_pred[i]) != int(gold_values[i]):

            print(
                f"\nStudy: {final_pred['StudyInstanceUID'].iloc[i]}"
            )
            print(f"Gold:  {int(gold_values[i])}")
            print(f"LLM1:  {p1.iloc[i]}")
            print(f"LLM2:  {p2[i]}")
            print(f"LLM3:  {p3[i]}")

In [ ]:
# FIND THE BEST METHOD FOR EACH ABNORMALITY
# LLM1 / LLM2 / LLM3 / majority combinations

from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import pandas as pd

methods = [
    "LLM1",
    "LLM2",
    "LLM3",
    "LLM1+LLM2",
    "LLM1+LLM3",
    "LLM2+LLM3",
    "LLM1+LLM2+LLM3"
]

all_results = []

# Use the same 58 gold studies
study_ids = final_pred["StudyInstanceUID"].reset_index(drop=True)

for target in target_cols:

    # -------------------------
    # Get predictions
    # -------------------------

    # LLM1
    p1 = llm1.set_index("StudyInstanceUID")[target]
    p1 = p1.reindex(study_ids).reset_index(drop=True)
    p1 = p1.where(p1.isin([0, 1]), np.nan)

    # LLM2
    p2 = llm2.set_index("StudyInstanceUID")[f"{target}__verdict"]
    p2 = p2.reindex(study_ids).reset_index(drop=True)

    p2 = np.where(
        p2 == "YES", 1,
        np.where(p2 == "NO", 0, np.nan)
    )

    p2 = pd.Series(p2)

    # LLM3
    p3 = llm3.set_index("StudyInstanceUID")[target]
    p3 = p3.reindex(study_ids).reset_index(drop=True)

    p3 = np.where(
        p3 > 0.5, 1,
        np.where(p3 < 0.5, 0, np.nan)
    )

    p3 = pd.Series(p3)

    # Gold
    y = gold_eval[target].reset_index(drop=True)

    predictions = {
        "LLM1": p1,
        "LLM2": p2,
        "LLM3": p3
    }

    # -------------------------
    # Individual LLMs
    # -------------------------

    for method in ["LLM1", "LLM2", "LLM3"]:

        pred = predictions[method]

        mask = y.notna() & pred.notna()

        if mask.sum() == 0:
            continue

        yt = y[mask].astype(int)
        yp = pred[mask].astype(int)

        all_results.append({
            "Abnormality": target,
            "Method": method,
            "Correct": (yt.values == yp.values).sum(),
            "Evaluated": len(yt),
            "Accuracy": accuracy_score(yt, yp),
            "F1": f1_score(yt, yp, zero_division=0)
        })

    # -------------------------
    # 2-LLM combinations
    # -------------------------

    pairs = [
        ("LLM1+LLM2", p1, p2),
        ("LLM1+LLM3", p1, p3),
        ("LLM2+LLM3", p2, p3)
    ]

    for method, a, b in pairs:

        # Only use cases where both give definite predictions
        mask = y.notna() & a.notna() & b.notna()

        if mask.sum() == 0:
            continue

        yt = y[mask].astype(int)

        # If they agree, use their answer
        # If they disagree, don't force a prediction
        yp = pd.Series(
            np.where(a[mask].values == b[mask].values,
                     a[mask].values,
                     np.nan)
        )

        valid = yp.notna()

        if valid.sum() == 0:
            continue

        yt = yt.reset_index(drop=True)[valid.values]
        yp = yp[valid].astype(int)

        all_results.append({
            "Abnormality": target,
            "Method": method,
            "Correct": (yt.values == yp.values).sum(),
            "Evaluated": len(yt),
            "Accuracy": accuracy_score(yt, yp),
            "F1": f1_score(yt, yp, zero_division=0)
        })

    # -------------------------
    # 3-LLM majority
    # -------------------------

    mask = y.notna() & (
        p1.notna() | p2.notna() | p3.notna()
    )

    majority_pred = []

    for i in range(len(y)):

        votes = []

        if pd.notna(p1[i]):
            votes.append(int(p1[i]))

        if pd.notna(p2[i]):
            votes.append(int(p2[i]))

        if pd.notna(p3[i]):
            votes.append(int(p3[i]))

        if len(votes) == 0:
            majority_pred.append(np.nan)

        elif votes.count(1) > votes.count(0):
            majority_pred.append(1)

        elif votes.count(0) > votes.count(1):
            majority_pred.append(0)

        else:
            majority_pred.append(np.nan)

    majority_pred = pd.Series(majority_pred)

    mask = y.notna() & majority_pred.notna()

    if mask.sum() > 0:

        yt = y[mask].astype(int)
        yp = majority_pred[mask].astype(int)

        all_results.append({
            "Abnormality": target,
            "Method": "LLM1+LLM2+LLM3",
            "Correct": (yt.values == yp.values).sum(),
            "Evaluated": len(yt),
            "Accuracy": accuracy_score(yt, yp),
            "F1": f1_score(yt, yp, zero_division=0)
        })


# ---------------------------------------
# Results
# ---------------------------------------

combination_results = pd.DataFrame(all_results)

# Best method by F1 for each abnormality
best_methods = (
    combination_results
    .sort_values(
        ["Abnormality", "F1", "Accuracy"],
        ascending=[True, False, False]
    )
    .groupby("Abnormality")
    .first()
    .reset_index()
)

print("BEST METHOD FOR EACH ABNORMALITY")
print("=" * 90)

print(
    best_methods[
        [
            "Abnormality",
            "Method",
            "Correct",
            "Evaluated",
            "Accuracy",
            "F1"
        ]
    ].to_string(index=False)
)

print("\n\nALL METHODS")
print("=" * 90)

print(
    combination_results[
        [
            "Abnormality",
            "Method",
            "Correct",
            "Evaluated",
            "Accuracy",
            "F1"
        ]
    ].to_string(index=False)
)

print("\n\nOVERALL")
print("=" * 90)

print("Macro F1 of selected methods:",
      best_methods["F1"].mean())

print("Mean Accuracy of selected methods:",
      best_methods["Accuracy"].mean())

In [ ]:
# COVERAGE-AWARE BEST METHOD
# Finds the best method per abnormality while maintaining minimum coverage

coverage_thresholds = [0.80, 0.85, 0.90, 0.95]

for threshold in coverage_thresholds:

    eligible = combination_results[
        (combination_results["Evaluated"] / 58) >= threshold
    ].copy()

    best = (
        eligible
        .sort_values(
            ["Abnormality", "F1", "Accuracy"],
            ascending=[True, False, False]
        )
        .groupby("Abnormality")
        .first()
        .reset_index()
    )

    print("\n" + "=" * 100)
    print(f"MINIMUM COVERAGE: {threshold*100:.0f}%")
    print("=" * 100)

    print(
        best[
            [
                "Abnormality",
                "Method",
                "Correct",
                "Evaluated",
                "Accuracy",
                "F1"
            ]
        ].to_string(index=False)
    )

    print("\nMacro F1:", best["F1"].mean())
    print("Mean Accuracy:", best["Accuracy"].mean())
    print("Mean Coverage:", (best["Evaluated"] / 58).mean())

In [ ]:
import pandas as pd
import numpy as np

target_cols = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

# --------------------------------------------------
# LLM1
# 0 = NO, 1 = YES
# Only exact 0/1 are treated as definite.
# --------------------------------------------------

llm1_clean = llm1[["StudyInstanceUID"] + target_cols].copy()

for col in target_cols:
    llm1_clean[col] = llm1_clean[col].where(
        llm1_clean[col].isin([0, 1]),
        np.nan
    )


# --------------------------------------------------
# LLM2
# YES = 1
# NO  = 0
# UNK = NaN
# --------------------------------------------------

llm2_clean = llm2[["StudyInstanceUID"]].copy()

for col in target_cols:

    llm2_clean[col] = np.where(
        llm2[col + "__verdict"] == "YES", 1,
        np.where(
            llm2[col + "__verdict"] == "NO", 0,
            np.nan
        )
    )


# --------------------------------------------------
# LLM3
# < 0.5 = NO
# > 0.5 = YES
# = 0.5 = uncertain
# --------------------------------------------------

llm3_clean = llm3[["StudyInstanceUID"] + target_cols].copy()

for col in target_cols:

    llm3_clean[col] = np.where(
        llm3[col] > 0.5, 1,
        np.where(
            llm3[col] < 0.5, 0,
            np.nan
        )
    )

print("LLM1:", llm1_clean.shape)
print("LLM2:", llm2_clean.shape)
print("LLM3:", llm3_clean.shape)

In [ ]:
pseudo = llm1_clean.merge(
    llm2_clean,
    on="StudyInstanceUID",
    how="outer",
    suffixes=("_llm1", "_llm2")
)

pseudo = pseudo.merge(
    llm3_clean,
    on="StudyInstanceUID",
    how="outer"
)

pseudo.shape

In [ ]:
final_labels = pd.DataFrame()

final_labels["StudyInstanceUID"] = pseudo["StudyInstanceUID"]


for col in target_cols:

    p1 = pseudo[col + "_llm1"]
    p2 = pseudo[col + "_llm2"]
    p3 = pseudo[col]

    # ---------------------------------------------
    # DEFAULT: LLM1
    # ---------------------------------------------

    final = p1.copy()


    # ---------------------------------------------
    # ACL
    # LLM1 + LLM3 when they agree
    # Otherwise LLM1
    # ---------------------------------------------

    if col == "ACL":

        agree = (
            p1.notna() &
            p3.notna() &
            (p1 == p3)
        )

        final.loc[agree] = p1.loc[agree]


    # ---------------------------------------------
    # MEDIAL MENISCUS
    # LLM1 + LLM2 when they agree
    # Otherwise LLM1
    # ---------------------------------------------

    elif col == "Medial Meniscus":

        agree = (
            p1.notna() &
            p2.notna() &
            (p1 == p2)
        )

        final.loc[agree] = p1.loc[agree]


    # ---------------------------------------------
    # FRACTURE
    # Majority of 3
    # If no majority, use LLM1
    # ---------------------------------------------

    elif col == "Fracture":

        for i in range(len(final)):

            votes = []

            if pd.notna(p1.iloc[i]):
                votes.append(int(p1.iloc[i]))

            if pd.notna(p2.iloc[i]):
                votes.append(int(p2.iloc[i]))

            if pd.notna(p3.iloc[i]):
                votes.append(int(p3.iloc[i]))

            if len(votes) >= 2:

                if votes.count(1) >= 2:
                    final.iloc[i] = 1

                elif votes.count(0) >= 2:
                    final.iloc[i] = 0

            elif len(votes) == 1:

                final.iloc[i] = votes[0]


    # ---------------------------------------------
    # ALL OTHER ABNORMALITIES
    # LLM1
    # ---------------------------------------------

    final_labels[col] = final


print(final_labels.shape)

print("\nMissing labels:")
print(final_labels[target_cols].isna().sum())

In [ ]:
for col in target_cols:

    print("\n", col)

    print(
        final_labels[col]
        .value_counts(dropna=False)
        .sort_index()
    )

In [ ]:
complete_studies = final_labels.dropna(
    subset=target_cols
).copy()

print("Total studies:", len(final_labels))
print("Complete 12-label studies:", len(complete_studies))
print(
    "Percentage complete:",
    len(complete_studies) / len(final_labels) * 100
)

In [ ]:
for col in target_cols:
    print(
        f"{col:20s} "
        f"YES = {int((final_labels[col] == 1).sum()):4d} | "
        f"NO = {int((final_labels[col] == 0).sum()):4d} | "
        f"MISSING = {int(final_labels[col].isna().sum()):4d}"
    )

In [ ]:
output_path = "/kaggle/working/final_pseudo_labels.csv"

final_labels.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Shape:", final_labels.shape)